# SmolLM2 Integrated Memory V3 — train, evaluate, and publish

**GPU runtime → Run all.** Use `smoke` first, then `balanced` for the research run.

This notebook trains the Integrated Memory V3 candidates for **HuggingFaceTB/SmolLM2-135M**, evaluates the complete hybrid model, and can publish the validation-selected checkpoint to Hugging Face with a reproducible model card.

The Hugging Face client is pinned to `huggingface_hub==0.36.2`, which is compatible with `transformers==4.57.6` and avoids the mixed-package `DeviceCodeError` import failure.

## Configurations

| Configuration | Changed layers | Purpose |
|---|---:|---|
| Original | none | untouched pretrained model |
| Attention conservative | 0,29 | matched adapted full-attention control |
| Attention expanded | 0,18,29 | larger full-attention control |
| Partition conservative | 0,29 | primary bounded-memory candidate |
| Partition expanded | 0,18,29 | test aggressive replacement |
| Sink conservative | 0,29 | no-training local/sink control |

Selection uses validation NLL only. Test results are read only after selection is locked.

In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

REPO_REF = "main"
REPO = Path(tempfile.mkdtemp(prefix="smollm2-integrated-v3-")) / "TinyCeNN-LM"
subprocess.run(["git", "clone", "--quiet", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO, check=True)

# Keep transformers and huggingface_hub on a known-compatible pair.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "transformers==4.57.6", "huggingface_hub==0.36.2", "datasets>=3,<5",
                "pytest", "nbformat", "pandas", "matplotlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"], check=True)

os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO), str(REPO / "src")])
sys.path[:0] = [str(REPO), str(REPO / "src")]

import torch
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Python:", sys.version.split()[0])


In [ ]:
PROFILE = "balanced" # @param ["smoke", "balanced", "extended"]
SAVE_TO_DRIVE = True # @param {type:"boolean"}
SEED = 2028 # @param {type:"integer"}

PROFILES = {
    "smoke": dict(train_contexts="32,64", test_contexts="32,64,128", block_size=8, features=16,
                  train_documents=4, validation_documents=2, test_documents=2, warm_documents=2,
                  warm_steps=2, joint_steps=4, eval_every=2, timing_documents=1, timing_repeats=1, decode_tokens=8),
    "balanced": dict(train_contexts="256,512,1024", test_contexts="256,512,1024,2048", block_size=32, features=64,
                     train_documents=128, validation_documents=16, test_documents=32, warm_documents=16,
                     warm_steps=100, joint_steps=300, eval_every=50, timing_documents=3, timing_repeats=3, decode_tokens=32),
    "extended": dict(train_contexts="512,1024,2048", test_contexts="512,1024,2048,4096", block_size=32, features=64,
                     train_documents=256, validation_documents=32, test_documents=64, warm_documents=32,
                     warm_steps=200, joint_steps=1000, eval_every=100, timing_documents=5, timing_repeats=5, decode_tokens=64),
}

if not torch.cuda.is_available() and PROFILE != "smoke":
    raise RuntimeError("Select a GPU runtime, or use smoke for a CPU pipeline check.")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/TinyCeNN/smollm2-integrated-v3")
else:
    BASE = Path("/content/smollm2-integrated-v3-results") if Path("/content").exists() else Path.cwd()/"v3-results"

BASE.mkdir(parents=True, exist_ok=True)
run_id = PROFILE + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT, LOG = BASE / run_id, BASE / (run_id + ".log")
config = dict(PROFILES[PROFILE], seed=SEED)
print(json.dumps(config, indent=2))
print("Results:", OUT)


## Optional exclusions

V3 already excludes the documented prior holdouts/hashes. Add older experiment manifests here when you want stricter cross-run exclusion.

In [ ]:
ADDITIONAL_MANIFESTS = []
UPLOAD_MANIFESTS = False # @param {type:"boolean"}

if UPLOAD_MANIFESTS:
    from google.colab import files
    folder = Path(tempfile.mkdtemp(prefix="v3-exclusions-"))
    for index, (name, content) in enumerate(files.upload().items()):
        value = json.loads(content)
        if not value.get("document_hashes"):
            raise ValueError(f"{name}: expected a manifest with document_hashes")
        path = folder / f"manifest-{index}.json"
        path.write_bytes(content)
        ADDITIONAL_MANIFESTS.append(str(path))

print("Additional manifests:", len(ADDITIONAL_MANIFESTS))


## Preflight and training

The CPU preflight checks gradients, frozen teacher weights, checkpoint reload, native precision selection, and full-sequence/cached equivalence before the expensive run.

In [ ]:
env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_integrated_memory.py"],
               cwd=REPO, env=env, check=True)


In [ ]:
command = [sys.executable, "-u", str(REPO / "scripts/benchmark_smollm2_integrated_memory.py"),
           "--output-dir", str(OUT)]
for key, value in config.items():
    command += ["--" + key.replace("_", "-"), str(value)]
for path in ADDITIONAL_MANIFESTS:
    command += ["--exclude-manifest", str(path)]

print(" ".join(command))
try:
    with LOG.open("w") as log:
        with subprocess.Popen(command, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"Run failed with code {status}; inspect {LOG}")
except BaseException as error:
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "failure_report.json").write_text(json.dumps({"error": str(error), "log": str(LOG)}, indent=2))
    raise
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG, OUT / "console.log")
        archive = shutil.make_archive(str(OUT) + "-results", "zip", root_dir=OUT)
        print("Archive:", archive)


## Results

Inspect the validation-selected model against the untouched Transformer, matched adapted-attention control, cache usage, and complete-model timing.

In [ ]:
import pandas as pd

summary = pd.read_csv(OUT / "integrated_summary.csv")
selection = json.loads((OUT / "selection.json").read_text())
selected = selection["selected"]
cols = [c for c in ["candidate", "context", "test_perplexity", "ppl_ratio", "adapted_ppl_ratio",
                       "before_joint_ppl_ratio", "prefill_speedup", "decode_speedup", "total_cache_ratio",
                       "beats_both_quality", "quality_preserving_efficiency_win", "selected_on_validation"]
        if c in summary.columns]
print("Locked validation selection:", selected)
display(summary[cols].sort_values(["candidate", "context"]).reset_index(drop=True))


# Publish the selected checkpoint to Hugging Face

This cell publishes the **exact validation-selected V3 checkpoint** plus its pinned SmolLM2 base revision, TinyCeNN source, benchmark evidence, loader, and model card.

### Before running
In Colab **Secrets** (key icon), add a write-enabled secret named `HF_TOKEN`.

The upload itself runs in a fresh Python subprocess. This avoids stale in-memory `huggingface_hub` modules and specifically avoids the `DeviceCodeError` mixed-version failure.

In [ ]:
# @title Export + publish selected V3 checkpoint
HF_REPO_ID = "vtava/SmolLM2-135M-CeNN-Partition-V3" # @param {type:"string"}
PRIVATE = False # @param {type:"boolean"}

import json, os, shutil, subprocess, sys
from pathlib import Path
import pandas as pd

# Repair the on-disk Hub package in case this runtime previously upgraded it to 1.x.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                "huggingface_hub==0.36.2"], check=True)

OUT = Path(OUT)
REPO = Path(REPO)
selection = json.loads((OUT / "selection.json").read_text())
manifest = json.loads((OUT / "manifest.json").read_text())
report = json.loads((OUT / "integrated_report.json").read_text())
selected = selection["selected"]
record = next(r for r in report["candidates"] if r["candidate"] == selected)
checkpoint = OUT / record["checkpoint"]
if not checkpoint.exists():
    raise FileNotFoundError(f"Selected checkpoint is missing: {checkpoint}")

args = manifest["args"]
PUBLISH = OUT / "huggingface_export"
if PUBLISH.exists(): shutil.rmtree(PUBLISH)
PUBLISH.mkdir(parents=True)
MODEL_FILE = f"{selected}.pt"
shutil.copy2(checkpoint, PUBLISH / MODEL_FILE)

adapter_config = {
    "format": "smollm2-integrated-memory-v3", "candidate": selected,
    "variant": record["variant"], "layers": record["layers"],
    "base_model": args["base_model"], "base_revision": manifest["model_revision"],
    "tinycenn_source_commit": manifest["source_commit"],
    "feature_dimension": int(args["features"]), "block_size": int(args["block_size"]),
    "sink_tokens": int(args["sinks"]), "validation_nll": record["validation_nll"],
    "validation_nll_before_joint": record["before_joint_validation_nll"],
    "trainable_parameters": record["trainable_parameters"], "joint_updates": record["joint_updates"],
    "joint_token_presentations": record["joint_token_presentations"],
    "transformers_version": manifest["transformers"], "training_precision": manifest["native_dtype"],
    "source_repository": "https://github.com/vtavakkoli/TinyCeNN-LM"
}
(PUBLISH / "adapter_config.json").write_text(json.dumps(adapter_config, indent=2))

pkg = PUBLISH / "tinycenn_lm"; pkg.mkdir(); (pkg / "__init__.py").write_text("")
for fn in ["integrated_memory.py", "optimized_memory.py"]:
    shutil.copy2(REPO / "src" / "tinycenn_lm" / fn, pkg / fn)
if (REPO / "LICENSE").exists(): shutil.copy2(REPO / "LICENSE", PUBLISH / "LICENSE-TinyCeNN-LM")

loader = [
"import json", "from pathlib import Path", "import torch",
"from huggingface_hub import snapshot_download",
"from transformers import AutoModelForCausalLM, AutoTokenizer",
"from tinycenn_lm.integrated_memory import restore_student", "",
"def load_model(repo_id, device=None, token=None):",
"    folder = Path(snapshot_download(repo_id, token=token))",
"    cfg = json.loads((folder / 'adapter_config.json').read_text())",
"    if torch.cuda.is_available():",
"        major, _ = torch.cuda.get_device_capability()",
"        dtype = torch.bfloat16 if major >= 8 else torch.float16",
"    else: dtype = torch.float32",
"    base = AutoModelForCausalLM.from_pretrained(cfg['base_model'], revision=cfg['base_revision'], torch_dtype=dtype, attn_implementation='sdpa', token=token).eval().requires_grad_(False)",
"    payload = torch.load(folder / '" + MODEL_FILE + "', map_location='cpu', weights_only=True)",
"    model = restore_student(base, payload).eval()",
"    if device is None: device = 'cuda' if torch.cuda.is_available() else 'cpu'",
"    model = model.to(device)",
"    tok = AutoTokenizer.from_pretrained(cfg['base_model'], revision=cfg['base_revision'], token=token)",
"    return model, tok", ""]
(PUBLISH / "load_model.py").write_text("\n".join(loader))
(PUBLISH / "requirements.txt").write_text(f"torch\ntransformers=={manifest['transformers']}\nhuggingface_hub==0.36.2\n")

bench = PUBLISH / "benchmark"; bench.mkdir()
for fn in ["manifest.json", "selection.json", "integrated_report.json", "validation_summary.csv",
           "integrated_summary.csv", "decision_table.csv", "test_document_nll.csv", "generation_examples.json",
           "partition_conservative_history.csv", "training_history.csv", "joint-training.png",
           "integrated-T256.png", "integrated-T512.png", "integrated-T1024.png", "integrated-T2048.png"]:
    src = OUT / fn
    if src.exists(): shutil.copy2(src, bench / fn)

summary = pd.read_csv(OUT / "integrated_summary.csv")
rows = summary[summary["candidate"] == selected].sort_values("context")
lines = ["| Context | Perplexity | PPL ratio | Cache ratio | Prefill | Decode |",
         "|---:|---:|---:|---:|---:|---:|"]
for _, r in rows.iterrows():
    lines.append(f"| {int(r['context'])} | {r['test_perplexity']:.3f} | {r['ppl_ratio']:.4f} | {r['total_cache_ratio']:.4f} | {r['prefill_speedup']:.3f}× | {r['decode_speedup']:.3f}× |")
last = rows.iloc[-1]
cache_reduction = (1-float(last["total_cache_ratio"]))*100
ppl_change = (float(last["ppl_ratio"])-1)*100

card=[]
def add(x=""): card.append(str(x))
for x in [
"---", "language:", "- en", "license: apache-2.0", "library_name: transformers", "pipeline_tag: text-generation",
"base_model:", f"- {args['base_model']}", "tags:", "- smollm2", "- cenn", "- recurrent-memory", "- hybrid-attention", "- memory-efficient", "- experimental", "---", "",
"# SmolLM2-135M CeNN Partition V3", "", "## TinyCeNN Integrated Memory", "",
f"This repository contains the validation-selected **`{selected}`** checkpoint from TinyCeNN-LM Integrated Memory V3.", "",
"## Architecture", "", f"- Base model: `{args['base_model']}`", f"- Exact base revision: `{manifest['model_revision']}`",
f"- TinyCeNN source commit: `{manifest['source_commit']}`", f"- Variant: `{record['variant']}`", f"- Replaced attention layers: `{record['layers']}`",
f"- Feature dimension: `{args['features']}`", f"- Block size: `{args['block_size']}`", f"- Sink tokens: `{args['sinks']}`",
f"- Trainable TinyCeNN parameters: `{record['trainable_parameters']:,}`", "",
"The remaining layers retain standard Transformer attention. Original pretrained embeddings, FFNs, norms and Q/K/V/O projections stay frozen.", "",
"## Selection", "", "The checkpoint was selected by validation NLL before held-out test evaluation.", "",
f"- Validation NLL before joint training: **{record['before_joint_validation_nll']:.6f}**", f"- Validation NLL after joint training: **{record['validation_nll']:.6f}**", "",
"## Held-out evaluation", "", *lines, "",
f"At **{int(last['context'])} tokens**, total cache is about **{cache_reduction:.1f}% lower**, with a perplexity change of **{ppl_change:+.2f}%** versus the untouched baseline.", "",
"The current PyTorch implementation is not yet faster than optimized SDPA; treat this primarily as a memory/architecture research result.", "",
"## Load", "", "```python", "from huggingface_hub import snapshot_download", "import sys", f"folder = snapshot_download('{HF_REPO_ID}')", "sys.path.insert(0, folder)", "from load_model import load_model", f"model, tokenizer = load_model('{HF_REPO_ID}')", "```", "",
"## Reproducibility", "", f"- Training documents: `{args['train_documents']}`", f"- Validation documents: `{args['validation_documents']}`", f"- Test documents: `{args['test_documents']}`", f"- Training contexts: `{args['train_contexts']}`", f"- Test contexts: `{args['test_contexts']}`", f"- Joint updates: `{record['joint_updates']}`", f"- Seed: `{args['seed']}`", f"- Precision: `{manifest['native_dtype']}`", f"- GPU: `{manifest['gpu']}`", "",
"## Limitations", "", f"Only layers **{record['layers']}** are replaced. This is not a fully attention-free model. The experiment uses one training seed and a limited held-out set; it does not establish universal superiority over Transformer attention.", "",
"## Source", "", "https://github.com/vtavakkoli/TinyCeNN-LM", "", "## License", "", "SmolLM2 base weights: Apache-2.0. TinyCeNN-LM source: MIT (included as `LICENSE-TinyCeNN-LM`).", "",
"## Status", "", "- 🧪 Experimental research checkpoint", "- ✅ Validation-selected", "- ✅ Near-original language-model quality", "- ✅ Lower cache at longer context", "- 🚧 Kernel optimization and larger-scale evaluation remain future work"]: add(x)
(PUBLISH / "README.md").write_text("\n".join(card), encoding="utf-8")

# Get a write token without importing huggingface_hub in this already-running process.
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    raise RuntimeError("Add a write-enabled Colab secret named HF_TOKEN, then rerun this cell.")

upload_script = r'''import os, huggingface_hub
from huggingface_hub import HfApi
repo=os.environ["TINY_REPO"]; folder=os.environ["TINY_FOLDER"]; token=os.environ["HF_TOKEN"]
print("huggingface_hub:", huggingface_hub.__version__)
api=HfApi(token=token)
api.create_repo(repo_id=repo, repo_type="model", private=os.environ["TINY_PRIVATE"]=="true", exist_ok=True, token=token)
api.upload_folder(repo_id=repo, repo_type="model", folder_path=folder, token=token, commit_message="Publish TinyCeNN Integrated Memory V3 selected checkpoint")
print("Published: https://huggingface.co/" + repo)
'''
env=os.environ.copy(); env.update(HF_TOKEN=HF_TOKEN, TINY_REPO=HF_REPO_ID, TINY_FOLDER=str(PUBLISH), TINY_PRIVATE=str(PRIVATE).lower())
r=subprocess.run([sys.executable, "-c", upload_script], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError("Hugging Face upload failed; see output above.")
print("✅ Published exact selected checkpoint:", selected)
print("https://huggingface.co/" + HF_REPO_ID)
